In [43]:
import sqlite3
import urllib.request
import pandas as pd
from itertools import permutations

In [44]:
url = "https://sq.io/testdata/sakila.db"
urllib.request.urlretrieve(url, "sakila.db")
print("Databse Downloaded Successfully !!")

Databse Downloaded Successfully !!


In [45]:
con = sqlite3.connect("sakila.db")

tables = [r[0] for r in con.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")]
print(len(tables), "tables:")
print(tables)

16 tables:
['actor', 'country', 'city', 'address', 'language', 'category', 'customer', 'film', 'film_actor', 'film_category', 'film_text', 'inventory', 'staff', 'store', 'payment', 'rental']


In [46]:
pd.read_sql("SELECT * FROM country LIMIT 50", con)

,country_id,country,last_update
0,1,Afghanistan,2020-02-15 06:59:27
1,2,Algeria,2020-02-15 06:59:27
2,3,American Samoa,2020-02-15 06:59:27
3,4,Angola,2020-02-15 06:59:27
4,5,Anguilla,2020-02-15 06:59:27
5,6,Argentina,2020-02-15 06:59:27
6,7,Armenia,2020-02-15 06:59:27
7,8,Australia,2020-02-15 06:59:27
8,9,Austria,2020-02-15 06:59:27
9,10,Azerbaijan,2020-02-15 06:59:27


In [47]:
def norm(v):
    # 1.0 aur 1 ko same value maano
    if isinstance(v, float) and v.is_integer():
        v = int(v)
    return str(v)

cols = {}   # (table, column) -> set of values
for t in tables:
    df = pd.read_sql(f'SELECT * FROM "{t}"', con)
    for c in df.columns:
        vals = set(map(norm, df[c].dropna()))
        if vals:
            cols[(t, c)] = vals

print(len(cols), "Columns Loaded Successfully")

83 Columns Loaded Successfully


In [48]:
# Initial Baseline Function
def discover_inds(cols):
    found = set()
    for a, b in permutations(cols, 2):
        if a[0] != b[0] and cols[a] <= cols[b]:   # different tables, A ⊆ B
            found.add((a, b))
    return found

found = discover_inds(cols)

print(len(found), "INDs Found")

596 INDs Found


In [49]:
#Finding Ground Truth
def primary_key(t):
    return [r[1] for r in con.execute(f'PRAGMA table_info("{t}")') if r[5] >= 1]

gt = set()
for t in tables:
    for r in con.execute(f'PRAGMA foreign_key_list("{t}")'):
        ref_table, from_col, to_col = r[2], r[3], r[4]
        if to_col is None:                      # kabhi-kabhi sirf PK implicit hota hai
            pk = primary_key(ref_table)
            to_col = pk[0] if pk else None
        if to_col and ref_table != t:
            gt.add(((t, from_col), (ref_table, to_col)))

print(len(gt), "declared FKs")
sorted(gt)[:22]

22 declared FKs


[(('address', 'city_id'), ('city', 'city_id')),
 (('city', 'country_id'), ('country', 'country_id')),
 (('customer', 'address_id'), ('address', 'address_id')),
 (('customer', 'store_id'), ('store', 'store_id')),
 (('film', 'language_id'), ('language', 'language_id')),
 (('film', 'original_language_id'), ('language', 'language_id')),
 (('film_actor', 'actor_id'), ('actor', 'actor_id')),
 (('film_actor', 'film_id'), ('film', 'film_id')),
 (('film_category', 'category_id'), ('category', 'category_id')),
 (('film_category', 'film_id'), ('film', 'film_id')),
 (('inventory', 'film_id'), ('film', 'film_id')),
 (('inventory', 'store_id'), ('store', 'store_id')),
 (('payment', 'customer_id'), ('customer', 'customer_id')),
 (('payment', 'rental_id'), ('rental', 'rental_id')),
 (('payment', 'staff_id'), ('staff', 'staff_id')),
 (('rental', 'customer_id'), ('customer', 'customer_id')),
 (('rental', 'inventory_id'), ('inventory', 'inventory_id')),
 (('rental', 'staff_id'), ('staff', 'staff_id')),
 

In [50]:
# Confusion Matrix
# Universe: alag tables ke saare possible (A, B) pairs, plus gt (taaki miss hua FK bhi count ho)
candidates = {(a, b) for a, b in permutations(cols, 2) if a[0] != b[0]} | gt

def confusion(found, positives, negatives=None):
    """positives = sahi INDs, negatives = confirm kiye hue galat INDs.
    negatives=None -> positives ke bahar sab kuch negative maano (strict view)."""
    if negatives is None:
        negatives = candidates - positives
    TP = len(found & positives)
    FN = len(positives - found)
    FP = len(found & negatives)
    TN = len((candidates - found) & negatives)
    P  = TP / (TP + FP) if TP + FP else 0
    R  = TP / (TP + FN) if TP + FN else 0
    F1 = 2 * P * R / (P + R) if P + R else 0
    matrix = pd.DataFrame([[TP, FN], [FP, TN]],
                          index=["Actual: sahi IND", "Actual: meaningless"],
                          columns=["Predicted: IND", "Predicted: not IND"])
    return matrix, P, R, F1

# View 1: Strict (sirf declared FKs ko sahi maano)
m, P, R, F1 = confusion(found, gt)
print("STRICT VIEW (declared FKs hi sahi)")
display(m)
print(f"Precision={P:.3f}  Recall={R:.3f}  F1={F1:.3f}")

# View 2: Manual labels ke saath (jaise-jaise label karte jao, yahan bharo)
meaningful_extra = set()   # extras jo tumhe sach mein meaningful lage
coincidental     = set()   # extras jo coincidence lage
m2, P2, R2, F12 = confusion(found, gt | meaningful_extra, coincidental)
print("\nLABELED VIEW (sirf tumhare labels ke hisaab se)")
display(m2)
print(f"Precision={P2:.3f}  Recall={R2:.3f}  F1={F12:.3f}")
print("Abhi tak unlabeled extras:", len(found - gt - meaningful_extra - coincidental))

STRICT VIEW (declared FKs hi sahi)


,Predicted: IND,Predicted: not IND
Actual: sahi IND,21,1
Actual: meaningless,575,5712


Precision=0.035  Recall=0.955  F1=0.068

LABELED VIEW (sirf tumhare labels ke hisaab se)


,Predicted: IND,Predicted: not IND
Actual: sahi IND,21,1
Actual: meaningless,0,0


Precision=1.000  Recall=0.955  F1=0.977
Abhi tak unlabeled extras: 575


In [51]:

cols[("city", "city_id")]

{'1',
 '10',
 '100',
 '101',
 '102',
 '103',
 '104',
 '105',
 '106',
 '107',
 '108',
 '109',
 '11',
 '110',
 '111',
 '112',
 '113',
 '114',
 '115',
 '116',
 '117',
 '118',
 '119',
 '12',
 '120',
 '121',
 '122',
 '123',
 '124',
 '125',
 '126',
 '127',
 '128',
 '129',
 '13',
 '130',
 '131',
 '132',
 '133',
 '134',
 '135',
 '136',
 '137',
 '138',
 '139',
 '14',
 '140',
 '141',
 '142',
 '143',
 '144',
 '145',
 '146',
 '147',
 '148',
 '149',
 '15',
 '150',
 '151',
 '152',
 '153',
 '154',
 '155',
 '156',
 '157',
 '158',
 '159',
 '16',
 '160',
 '161',
 '162',
 '163',
 '164',
 '165',
 '166',
 '167',
 '168',
 '169',
 '17',
 '170',
 '171',
 '172',
 '173',
 '174',
 '175',
 '176',
 '177',
 '178',
 '179',
 '18',
 '180',
 '181',
 '182',
 '183',
 '184',
 '185',
 '186',
 '187',
 '188',
 '189',
 '19',
 '190',
 '191',
 '192',
 '193',
 '194',
 '195',
 '196',
 '197',
 '198',
 '199',
 '2',
 '20',
 '200',
 '201',
 '202',
 '203',
 '204',
 '205',
 '206',
 '207',
 '208',
 '209',
 '21',
 '210',
 '211',
 '212',


In [52]:
#print("Extra INDs (Not FK):", sorted(found - gt))
print("Missed FKs        :", len(gt - found))
print("Missed FKs        :", sorted(gt - found))

Missed FKs        : 1
Missed FKs        : [(('film', 'original_language_id'), ('language', 'language_id'))]


# Advanced Baselines: unary IND discovery (De Marchi, SPIDER, BINDER-style)
Upar wala `discover_inds` ek naive pairwise check hai. Neeche teen research-style algorithms hain.
Sabka output naive wale ke **bilkul same** hona chahiye (correctness check last cell mein). Sirf kaam karne ka tarika aur speed alag hai.

Common rule: agar value `v` sirf columns `group` mein hai, to un columns ka koi bhi parent `group` ke bahar nahi ho sakta: `refs[X] = refs[X] ∩ group`.

In [53]:
import re, time, heapq
from collections import defaultdict

NUM_RE = re.compile(r"^-?\d+(\.\d+)?$")

def column_stats(cols):
    """Har column ke liye: type, distinct count, min, max (pruning ke liye)."""
    stats = {}
    for c, vals in cols.items():
        if all(NUM_RE.match(v) for v in vals):
            nums = [float(v) for v in vals]
            stats[c] = ("num", len(vals), min(nums), max(nums))
        else:
            stats[c] = ("str", len(vals), min(vals), max(vals))
    return stats

def initial_candidates(cols, prune=True):
    """refs[A] = un columns B ka set jo A ke parent ho sakte hain.
    Pruning rules (sab lossless hain, koi sahi IND nahi hatata):
      1. same table skip
      2. A mein non-numeric value hai aur B poora numeric -> A ⊄ B
      3. A ke distinct values B se zyada -> A ⊄ B
      4. A ka min/max B ki range ke bahar -> A ⊄ B (same type hone par)"""
    st = column_stats(cols)
    refs = {}
    for a in cols:
        refs[a] = set()
        for b in cols:
            if a[0] == b[0]:
                continue
            if prune:
                ta, na, lo_a, hi_a = st[a]
                tb, nb, lo_b, hi_b = st[b]
                if ta == "str" and tb == "num":
                    continue
                if na > nb:
                    continue
                if ta == tb and (lo_a < lo_b or hi_a > hi_b):
                    continue
            refs[a].add(b)
    return refs

def to_pairs(refs):
    return {(a, b) for a, bs in refs.items() for b in bs}

In [54]:
# 1) Naive (reference) - har candidate pair par set-subset check
def naive(cols, prune=False):
    refs = initial_candidates(cols, prune)
    return {(a, b) for a, bs in refs.items() for b in bs if cols[a] <= cols[b]}

In [55]:
# 2) De Marchi - inverted index: value -> columns jisme woh value hai
def demarchi(cols, prune=True):
    refs = initial_candidates(cols, prune)
    inverted = defaultdict(set)
    for c, vals in cols.items():
        for v in vals:
            inverted[v].add(c)
    for group in inverted.values():
        for c in group:
            if refs[c]:                 # jinke candidates khatam, unhe skip
                refs[c] &= group
    return to_pairs(refs)

In [56]:
# 3) SPIDER - har column sort karo, sabko ek saath merge karte hue padho
def spider(cols, prune=True):
    refs = initial_candidates(cols, prune)
    iters = {c: iter(sorted(v)) for c, v in cols.items()}
    heap = []
    for c, it in iters.items():
        v = next(it, None)
        if v is not None:
            heap.append((v, c))
    heapq.heapify(heap)                 # min-heap: sabse chhoti value upar

    while heap:
        v = heap[0][0]
        group = set()
        while heap and heap[0][0] == v: # same value wale saare columns
            group.add(heapq.heappop(heap)[1])
        for c in group:
            if refs[c]:
                refs[c] &= group
        for c in group:                 # in columns ki agli value push karo
            nxt = next(iters[c], None)
            if nxt is not None:
                heapq.heappush(heap, (nxt, c))
    return to_pairs(refs)

In [57]:
# 4) BINDER (simplified, in-memory): Phase 1 partition (hash buckets), Phase 2 validate bucket-by-bucket
def binder(cols, prune=True, n_buckets=16):
    refs = initial_candidates(cols, prune)

    # Phase 1: Partition
    buckets = [defaultdict(set) for _ in range(n_buckets)]   # bucket -> col -> values
    for c, vals in cols.items():
        for v in vals:
            buckets[hash(v) % n_buckets][c].add(v)

    # Phase 2: Validate, ek bucket ek baar
    for b in buckets:
        inverted = defaultdict(set)
        for c, vals in b.items():
            for v in vals:
                inverted[v].add(c)
        for group in inverted.values():
            for c in group:
                if refs[c]:
                    refs[c] &= group
        if not any(refs.values()):      # early termination: koi candidate bacha hi nahi
            break
    return to_pairs(refs)

In [58]:
# Pruning ka asar: candidates pehle vs baad
n_all    = sum(len(v) for v in initial_candidates(cols, prune=False).values())
n_pruned = sum(len(v) for v in initial_candidates(cols, prune=True).values())
print(f"Candidates without pruning: {n_all}")
print(f"Candidates after pruning  : {n_pruned}  ({100*(1-n_pruned/n_all):.1f}% hataye)")

Candidates without pruning: 6308
Candidates after pruning  : 1441  (77.2% hataye)


In [59]:
# Correctness check + timing + recall (ground truth gt se)
algos = {
    "naive (no prune)": lambda: naive(cols, prune=False),
    "naive + pruning":  lambda: naive(cols, prune=True),
    "De Marchi":        lambda: demarchi(cols),
    "SPIDER":           lambda: spider(cols),
    "BINDER (simpl.)":  lambda: binder(cols),
}

results, rows = {}, []
for name, f in algos.items():
    t0 = time.perf_counter()
    results[name] = f()
    dt = time.perf_counter() - t0
    r = results[name]
    rows.append({"algorithm": name, "INDs": len(r),
                 "recall vs FK": round(len(r & gt) / len(gt), 2),
                 "time (s)": round(dt, 4)})

reference = results["naive (no prune)"]
for row in rows:
    row["same as naive"] = results[row["algorithm"]] == reference
pd.DataFrame(rows)

,algorithm,INDs,recall vs FK,time (s),same as naive
0,naive (no prune),596,0.95,0.0364,True
1,naive + pruning,596,0.95,0.0365,True
2,De Marchi,596,0.95,0.0799,True
3,SPIDER,596,0.95,0.1498,True
4,BINDER (simpl.),596,0.95,0.0875,True
